# 02 – Hypothesis Testing & A/B Analysis

In this notebook we statistically test the following null hypotheses:

1. There are no risk differences across provinces.
2. There are no risk differences between zip codes.
3. There is no significant margin (profit) difference between zip codes.
4. There is no significant risk difference between women and men.

Risk is measured by:
- Claim frequency (`HasClaim`)
- Claim severity (`TotalClaims` where `HasClaim == 1`)
- Margin (`TotalPremium - TotalClaims`)


In [2]:
import os
import sys

# Make Python see the project root (one level above "notebooks")
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 100)


## 1. Load the processed dataset

In [3]:
DATA_PATH_PROCESSED = os.path.join("..", "data", "processed", "insurance_clean.csv")

df = pd.read_csv(DATA_PATH_PROCESSED)

df.head()


C:\Users\user\AppData\Local\Temp\ipykernel_12520\620266888.py:3: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH_PROCESSED)


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,MaritalStatus,Gender,Country,Province,PostalCode,MainCrestaZone,SubCrestaZone,ItemType,mmcode,VehicleType,RegistrationYear,make,Model,Cylinders,cubiccapacity,kilowatts,bodytype,NumberOfDoors,VehicleIntroDate,CustomValueEstimate,AlarmImmobiliser,TrackingDevice,CapitalOutstanding,NewVehicle,WrittenOff,Rebuilt,Converted,CrossBorder,NumberOfVehiclesInFleet,SumInsured,TermFrequency,CalculatedPremiumPerTerm,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin,HasClaim
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,0.01,Monthly,25.0000,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.00,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0.0,512.848070,0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,Not specified,Not specified,South Africa,Gauteng,1459,Rand East,Rand East,Mobility - Motor,44069150.0,Passenger Vehicle,2004,MERCEDES-BENZ,E 240,6.0,2597.0,130.0,S/D,4.0,6/2002,119300.0,Yes,No,119300,More than 6 months,NaN,NaN,NaN,NaN,NaN,119300.00,Monthly,584.6468,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,0


## 2. Helper: print test result

In [4]:
def print_test_result(name, statistic, p_value, alpha=0.05):
    print(f"=== {name} ===")
    print(f"Statistic: {statistic:.4f}")
    print(f"P-value:   {p_value:.6f}")
    if p_value < alpha:
        print(f"Decision: Reject H0 at alpha={alpha} (evidence of a difference)\n")
    else:
        print(f"Decision: Fail to reject H0 at alpha={alpha} (no strong evidence)\n")


## 3. Risk differences across provinces

In [5]:
if {"Province", "HasClaim"}.issubset(df.columns):
    contingency = pd.crosstab(df["Province"], df["HasClaim"])

    chi2, p, dof, expected = stats.chi2_contingency(contingency)

    print("Contingency table (Province x HasClaim):")
    display(contingency.head())

    print_test_result("Chi-square: Claim Frequency by Province", chi2, p)
else:
    print("Column 'Province' or 'HasClaim' not found in the dataset.")


Contingency table (Province x HasClaim):


HasClaim,0,1
Province,,
Eastern Cape,30286,50
Free State,8088,11
Gauteng,392543,1322
KwaZulu-Natal,169298,483
Limpopo,24769,67


=== Chi-square: Claim Frequency by Province ===
Statistic: 104.1909
P-value:   0.000000
Decision: Reject H0 at alpha=0.05 (evidence of a difference)



## 4. Risk & margin differences between zip codes

In [6]:
if "PostalCode" in df.columns:
    top_zips = df["PostalCode"].value_counts().head(20).index
    df_zip = df[df["PostalCode"].isin(top_zips)].copy()

    print("Top 20 PostalCodes by count:")
    print(df_zip["PostalCode"].value_counts().head())
else:
    print("Column 'PostalCode' not found.")


Top 20 PostalCodes by count:
PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
Name: count, dtype: int64


## 5. Risk differences between women and men

In [7]:
if {"Gender", "HasClaim"}.issubset(df.columns):
    contingency_gender = pd.crosstab(df["Gender"], df["HasClaim"])
    chi2_gender, p_gender, dof_gender, expected_gender = stats.chi2_contingency(contingency_gender)

    print("Contingency table (Gender x HasClaim):")
    display(contingency_gender)

    print_test_result("Chi-square: Claim Frequency by Gender", chi2_gender, p_gender)
else:
    print("Columns 'Gender' or 'HasClaim' not found.")


Contingency table (Gender x HasClaim):


HasClaim,0,1
Gender,,
Female,6741,14
Male,42723,94
Not specified,938324,2666


=== Chi-square: Claim Frequency by Gender ===
Statistic: 7.2559
P-value:   0.026570
Decision: Reject H0 at alpha=0.05 (evidence of a difference)

